# SmolVLA × LIBERO-plus LoRA 再学習 — PARC2026 予選 track1 向け

2026-08-07 版。**前回（`smolvla_libero_spatial_lora.ipynb`）の失敗を全部潰した構成。**

## 前回と何が違うか

| 項目 | 前回（成功率 0%） | 今回 |
|---|---|---|
| **バッチサイズ** | **1** | **32**（GPUに応じ自動調整＋ステップ数で補償） |
| **学習率** | **3e-4** | **1e-4**（ベースモデルと同じ） |
| **学習データ** | spatial 10タスク × 5話 = **50話** | **track1 の4タスク種** × 60話 = **240話** |
| **実効サンプル数** | 3,000 | **128,000** |
| **ベースとの比較評価** | ⚠️ **一度も実行しなかった** | **必須工程として組み込み。飛ばせない** |
| カメラ構成 | front + wrist（2台） | **同じ**（`empty_cameras=0` を維持） |

> **前回の最大の失敗は、比較対象を測らなかったこと。** 「改善したはず」を検証しないと、悪化していても分からない。今回はセクション12を飛ばすと最後まで進めない構成にしてある。

## 前提

- **ランタイム: GPU（A100 推奨）** — `ランタイム > ランタイムのタイプを変更 > A100`
- バッチ32を回すには **40GB級** が要る。それ未満なら自動で下げ、**ステップ数を増やして総サンプル数を一定に保つ**
- 所要時間の目安: A100 で **学習 1〜2時間 + 比較評価 1〜2時間**

## ⚠️ 実行上の注意

- **セクション3の直後にエラーが出たら「ランタイム > セッションを再起動」**して、セクション1から流し直す。モジュールキャッシュが原因（`/content` のファイルは消えない）
- 「接続を解除して削除」は押さないこと。**それはファイルごと消える**

## 1. ランタイムを確認する

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_LEROBOT_HOME"] = "/content/lerobot_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 以上が必要です。")

if not torch.cuda.is_available():
    raise RuntimeError("GPU ランタイムを選択してください。")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU     : {GPU_NAME}")
print(f"VRAM    : {GPU_GIB:.1f} GiB")
print(f"bfloat16: {'対応' if torch.cuda.is_bf16_supported() else '非対応（float16 で回す）'}")

## 2. システムパッケージを準備する

In [ ]:
def run_quiet(command: list[str], *, check: bool = True) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])
    return result


run_quiet(["apt-get", "update", "-qq"])
run_quiet([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "git", "unzip", "libgl1", "libglib2.0-0",
    "libsm6", "libxext6", "libexpat1",
    "libfontconfig1-dev", "libmagickwand-dev",
])

print("System packages ready.")

## 3. LeRobot を導入する

`v0.6.0` を固定。**`torchao` は必ず消す**（SmolVLA のインポートと衝突する）。

In [ ]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet([sys.executable, "-m", "pip", "uninstall", "-y", "lerobot", "torchao"], check=False)
shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet([
    "git", "clone", "--quiet", "--depth", "1", "--branch", LEROBOT_TAG,
    "https://github.com/huggingface/lerobot.git", str(LEROBOT_DIR),
])

# bf16 非対応 GPU（T4 など）では fp16 に落とす
smolvlm_source = LEROBOT_SRC / "lerobot" / "policies" / "smolvla" / "smolvlm_with_expert.py"
if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace('torch_dtype="bfloat16",', 'torch_dtype="float16",', 1)
    smolvlm_source.write_text(source, encoding="utf-8")

# 学習ログを静かにする
train_script = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_train.py"
source = train_script.read_text(encoding="utf-8")
source = source.replace("logging.info(pformat(cfg.to_dict()))", "logging.debug(pformat(cfg.to_dict()))", 1)
source = source.replace("disable=inside_slurm(),", "disable=True,", 1)
train_script.write_text(source, encoding="utf-8")

run_quiet([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
           "-e", f"{LEROBOT_DIR}[training,smolvla,peft]"])
run_quiet([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

for module_name in list(sys.modules):
    if module_name.split(".")[0] in {"lerobot", "torchao"}:
        del sys.modules[module_name]

sys.path = [item for item in sys.path if item not in {str(LEROBOT_DIR), str(LEROBOT_SRC)}]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchao の削除に失敗しました。ランタイムを再起動してやり直してください。")

import lerobot
import peft

if LEROBOT_SRC.resolve() not in Path(lerobot.__file__).resolve().parents:
    raise RuntimeError("LeRobot の読込先が想定と違います。")

print("LeRobot ready.")

## 4. 設定 ← **ここが前回との違いの本体**

### 学習対象タスク

PARC2026 予選 track1 の4タスクは、基本タスクとしてはこの4種：

| # | track1 のタスク名 | 言語指示 |
|---|---|---|
| 0 | `..._black_bowl_in_the_top_drawer_..._table_2` | pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate |
| 1 | `..._tomato_sauce_..._basket_table_27` | pick up the tomato sauce and place it in the basket |
| 2 | `..._milk_..._basket_light_15` | pick up the milk and place it in the basket |
| 3 | `put_the_bowl_on_the_stove_light_11` | put the bowl on the stove |

**前回は「黒いボウル→皿」の10タスクだけを学習していた。** カゴもコンロも一度も見せていない。

### バッチサイズの自動調整

バッチ32はVRAM 40GB級が要る。小さいGPUでは自動で下げ、**総サンプル数 `TOTAL_SAMPLES` が一定になるようステップ数を増やす**。

In [ ]:
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = "7bb70aa5bc92b82c9239142775d3a173103567ff"

VLM_REPO = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"

# ---- track1 の4タスク種（言語指示。大文字小文字・記号は正規化して照合する）----
TARGET_TASK_NAMES = [
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the tomato sauce and place it in the basket",
    "pick up the milk and place it in the basket",
    "put the bowl on the stove",
]

# ★ 学習に使うタスクの範囲
#   4種だけを厚く学習しても未知タスクには効かない。データセットの全タスクを
#   使い、物体・配置・言い回しの多様性そのものを増やす。
# ⚠️ 2026-08-09 実測: 全40タスクへ広げる（v4）は公式 0.13178 → 0.06948 と半減した。
#    v4 では track1 の4タスクが 60話→10話（6分の1）に減っていた。
#      多様性はタスク種ではなく、**同じタスクの摂動変種**で稼ぐべき。
USE_ALL_TASKS = False                 # v4 で否定された。4種に戻す
TRAIN_EPISODES_PER_TASK = 60          # v1 と同じ本数

# ★ 2026-08-09 の学習データ監査で判明したこと
#   v1 が学習した 240 エピソードのうち、**行動列がユニークなのは 124 本だけ**
#   （重複率 48.3%）。LIBERO-plus は1本のデモに複数の照明・テクスチャ摂動を
#   掛けて増やしており、エピソード数＝軌道の多様性ではない。
#   重複グループ内の画像は明度が大きく違う（中央 0.213）＝ light/table 系の摂動。
#   → 同じ把持軌道を2回暗記させる代わりに、別の軌道を入れたほうが情報量が多い。
DEDUP_TRAJECTORIES = True             # 行動列が同一のものは1本だけ採用する

# ★ 取り落とし軌道の除外
#   グリッパの開閉が4回＝掴み直し＝一度落としている疑い。
#   ⚠️ ただし drawer 系は「取っ手を掴む→放す→ボウルを掴む→放す」で4回が正常。
#      単純な pick-and-place のみ除外する（tomato sauce が 60本中12本＝20%）。
DROP_REGRASP = True

# ---- 学習ハイパーパラメータ ----
USE_AUGMENTATION = False              # ⚠️ 6種全部の拡張は実測で悪化した（上のコメント参照）
# ⚠️ v3 は「話数100」と「256kサンプル」を同時に変えて悪化した（ローカル 0.95→0.75）。
#    どちらが原因か切り分けていない。v5 では話数だけ動かし、サンプル数は v1 と同じに固定する。
TOTAL_SAMPLES = 128_000               # v1 と同じ。v3 の 256,000 は悪化した
LEARNING_RATE = 1e-4                  # 前回 3e-4 → ベースモデルと同じ 1e-4 に
FINAL_LEARNING_RATE = 1e-5
WARMUP_STEPS = 500
LORA_R = 16
LORA_ALPHA = 16
SEED = 42
LOG_FREQ = 100

# ---- VRAM からバッチサイズを決め、ステップ数で補償する ----
if GPU_GIB >= 38:
    BATCH_SIZE = 32
elif GPU_GIB >= 20:
    BATCH_SIZE = 16
elif GPU_GIB >= 14:
    BATCH_SIZE = 8
else:
    BATCH_SIZE = 4

STEPS = max(1000, TOTAL_SAMPLES // BATCH_SIZE)
SAVE_FREQ = STEPS

MIXED_PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

# ---- 比較評価の設定（セクション12で使う）----
EVAL_EPISODES_PER_TASK = 5    # 4タスク×5×2モデル=40話 ≈ 1時間
EVAL_N_ACTION_STEPS = 5       # ★ 実運用と揃える（設定ファイルの既定50は壊れた領域）
EVAL_SEED = 2026

# ---- 出力先 ----
OUTPUT_DIR = Path("/content/outputs/smolvla_libero_plus_t1_lora_v6")
MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_t1_lora_v6_merged")
BASELINE_MODEL_DIR = Path("/content/smolvla_libero_plus_baseline")
BASE_EVAL_DIR = Path("/content/eval_base")
FINETUNED_EVAL_DIR = Path("/content/eval_finetuned")
COMPARISON_CSV_PATH = Path("/content/comparison.csv")
MERGED_ZIP_PATH = Path("/content/t1_lora_v6_merged.zip")

print(f"GPU           : {GPU_NAME} ({GPU_GIB:.1f} GiB)")
print(f"batch_size    : {BATCH_SIZE}      ← 前回は 1")
print(f"steps         : {STEPS:,}")
print(f"実効サンプル数: {BATCH_SIZE * STEPS:,}  ← 前回は 3,000")
print(f"learning_rate : {LEARNING_RATE}   ← 前回は 3e-4")
print(f"mixed_precision: {MIXED_PRECISION}")
print()
if BATCH_SIZE < 32:
    print("⚠️ VRAM が足りないためバッチを下げました。ステップ数で総量は補っていますが、")
    print("   勾配のばらつきは大きくなります。可能なら A100 を選び直してください。")

## 5. HuggingFace からの取得を用意する（429 リトライ付き）

In [ ]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import HfHubHTTPError, LocalEntryNotFoundError

T = TypeVar("T")


def run_hf_with_retry(operation: Callable[[], T]) -> T:
    last_error: BaseException | None = None
    for attempt in range(6):
        try:
            return operation()
        except (HfHubHTTPError, httpx.HTTPStatusError) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)
            if status != 429 and "429" not in str(error):
                raise
            if attempt == 5:
                break
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(headers.get("Retry-After", 15)) + 1
            except (TypeError, ValueError):
                delay = min(120, 15 * (2**attempt) + random.random())
            time.sleep(delay)
    raise RuntimeError("Hugging Face からの取得に失敗しました。") from last_error


def cached_or_downloaded_snapshot(
    repo_id: str, revision: str, *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        return Path(snapshot_download(
            repo_id=repo_id, revision=revision, token=False,
            allow_patterns=allow_patterns, ignore_patterns=ignore_patterns,
            local_files_only=True,
        ))
    except (LocalEntryNotFoundError, FileNotFoundError):
        return Path(run_hf_with_retry(lambda: snapshot_download(
            repo_id=repo_id, revision=revision, token=False,
            allow_patterns=allow_patterns, ignore_patterns=ignore_patterns,
            max_workers=1,
        )))


print("Download helpers ready.")

## 6. 学習データを選ぶ — track1 の4タスク種

データセットのタスク名と `TARGET_TASK_NAMES` を正規化して突き合わせる。
**見つからなければ候補を表示して止まる**（前回のように黙って進まない）。

In [ ]:
import difflib
import re
from collections import defaultdict

import numpy as np
import pandas as pd

from lerobot.datasets.dataset_metadata import LeRobotDatasetMetadata


def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value
    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass
    return str(value)


def choose_evenly_spaced(episode_indices: list[int], count: int) -> list[int]:
    """均等に間引く。要求数がある分より多ければ全部返す。"""
    if count >= len(episode_indices):
        return list(episode_indices)
    positions = [round(i * (len(episode_indices) - 1) / (count - 1)) for i in range(count)]
    return [episode_indices[p] for p in sorted(set(positions))]


dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(DATASET_REPO, revision=DATASET_REVISION)
)

task_to_episodes: dict[str, list[int]] = defaultdict(list)
for episode_index, task_cell in enumerate(dataset_metadata.episodes["tasks"]):
    task_to_episodes[task_name_from_cell(task_cell)].append(int(episode_index))

available_by_normalized = {normalize_task_name(t): t for t in task_to_episodes}
print(f"データセットのタスク数: {len(task_to_episodes)}")

selected_by_task: dict[str, list[int]] = {}
missing: list[str] = []

wanted: list[str] = []                       # track1 の4種は必ず入れる
for task_name in TARGET_TASK_NAMES:
    actual = available_by_normalized.get(normalize_task_name(task_name))
    if actual is None:
        missing.append(task_name)
    else:
        wanted.append(actual)

if USE_ALL_TASKS:
    for t in sorted(task_to_episodes):       # 残り全部を追加
        if t not in wanted:
            wanted.append(t)
    print(f"★ 全タスク学習: {len(wanted)} タスク")
else:
    print(f"track1 の {len(wanted)} タスクのみ")

# --- 行動統計から「軌道の署名」を作る -------------------------------------
#   検証済み: min/max/mean/std/count の連結ハッシュは、行動列そのものの
#   ハッシュと 100% 一致した（240話・124軌道で確認）。
#   これによりデータ本体（数百MB）を落とさずに重複を判定できる。
import hashlib

def _traj_signature(row) -> str:
    v = np.concatenate([np.asarray(row[f"stats/action/{k}"]).ravel()
                        for k in ("min", "max", "mean", "std", "count")])
    return hashlib.md5(np.round(v, 6).tobytes()).hexdigest()[:12]

def _n_grip_flips(row) -> int:
    """グリッパ指令の符号反転回数の下限を統計から推定する。
    正常な pick-and-place は 開→閉→開 の2回。
    ⚠️ 統計量からは正確な回数が出ないので、ここでは使わず
       action の std が高いものを候補として見るだけに留める。"""
    return -1

# ⚠️ LeRobotDatasetMetadata.episodes には stats/action/* が無いので、
#    メタデータの parquet を直接取ってくる（約15MB。データ本体は不要）。
_sig = {}
if DEDUP_TRAJECTORIES:
    from huggingface_hub import snapshot_download as _snap
    _meta_dir = Path(run_hf_with_retry(lambda: _snap(
        repo_id=DATASET_REPO, revision=DATASET_REVISION, repo_type="dataset",
        allow_patterns=["meta/episodes/**"], token=False, max_workers=1)))
    _parts = sorted(_meta_dir.rglob("*.parquet"))
    print(f"エピソード表: {len(_parts)} ファイル")
    _ep_df = pd.concat([pd.read_parquet(p) for p in _parts], ignore_index=True)
    _need = [f"stats/action/{k}" for k in ("min", "max", "mean", "std", "count")]
    _missing = [c for c in _need if c not in _ep_df.columns]
    if _missing:
        print(f"⚠️ 必要な列がありません: {_missing}")
        print(f"   利用可能: {[c for c in _ep_df.columns if 'action' in c]}")
        DEDUP_TRAJECTORIES = False
    else:
        _sig = {int(r["episode_index"]): _traj_signature(r)
                for _, r in _ep_df.iterrows()}
        print(f"軌道署名を計算: {len(_sig)} エピソード / "
              f"ユニーク {len(set(_sig.values()))}")

for actual in wanted:
    pool = task_to_episodes[actual]
    if DEDUP_TRAJECTORIES:
        seen, uniq = set(), []
        for e in pool:                       # 若い episode_index を代表に採る
            s = _sig.get(int(e))
            if s is None or s in seen:
                continue
            seen.add(s); uniq.append(e)
        n_before = len(pool)
        pool = uniq
        print(f"  重複排除 {n_before:4d} → {len(pool):4d} 軌道  {actual[:46]}")
    selected_by_task[actual] = choose_evenly_spaced(pool, TRAIN_EPISODES_PER_TASK)

if missing:
    print("\n❌ 見つからなかったタスク:")
    for task_name in missing:
        print(f"\n  「{task_name}」")
        close = difflib.get_close_matches(
            normalize_task_name(task_name), available_by_normalized.keys(), n=5, cutoff=0.5
        )
        for c in close:
            print(f"      候補: {available_by_normalized[c]}")
    raise RuntimeError(
        "TARGET_TASK_NAMES を上の候補に合わせて修正し、セクション4から流し直してください。"
    )

EPISODE_INDICES = sorted(i for v in selected_by_task.values() for i in v)

print("\n=== 学習に使うエピソード（track1 の4種）===")
t1 = {available_by_normalized.get(normalize_task_name(t)) for t in TARGET_TASK_NAMES}
for task_name, indices in selected_by_task.items():
    if task_name in t1:
        print(f"  {len(indices):4d}話  (在庫 {len(task_to_episodes[task_name]):5d})  {task_name[:66]}")
others = [k for k in selected_by_task if k not in t1]
if others:
    n_oth = sum(len(selected_by_task[k]) for k in others)
    print(f"\n  その他 {len(others)} タスク: 計 {n_oth} 話")
    for k in others[:5]:
        print(f"      {len(selected_by_task[k]):3d}話  {k[:66]}")
    if len(others) > 5:
        print(f"      … 他 {len(others)-5} タスク")

print(f"\n  ★ 合計 {len(EPISODE_INDICES)} 話 / {len(selected_by_task)} タスク")
print("    参考: v1 = 240話/4タスク・128k（公式 0.13178・最良）")
print("          v2 = 画像拡張6種（公式 0.08735）/ v4 = 40タスク種（公式 0.06948）どちらも悪化")

if len(EPISODE_INDICES) > 2000:
    print("\n⚠️ エピソードが多く、データの初回取得に時間がかかります。")
    print("   TRAIN_EPISODES_PER_TASK を下げるか USE_ALL_TASKS=False に。")

## 7. ベース重みを取得する

In [ ]:
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    ignore_patterns=["eval/*", "*.mp4"],      # 評価動画は不要
)
print(f"ベースモデル: {BASE_MODEL_LOCAL}")
for f in sorted(BASE_MODEL_LOCAL.glob("*")):
    if f.is_file():
        print(f"  {f.stat().st_size/1e6:9.2f} MB  {f.name}")

## 8. LoRA 学習を実行する

**重要なフラグ**

| フラグ | 意味 |
|---|---|
| `--policy.input_features=null` `--policy.output_features=null` | データセットから入出力の形を作り直す |
| `--policy.empty_cameras=0` | **カメラ2台構成にする**（ベースは3台+空2枚を期待していて、PARCの2台と合わない） |
| `--policy.freeze_vision_encoder=true` | 視覚エンコーダは凍結 |
| `--policy.train_expert_only=true` | action expert だけ学習 |

In [ ]:
import re
from collections import deque

episodes_json = "[" + ",".join(map(str, EPISODE_INDICES)) + "]"

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    # 画像拡張（既定は無効）。
    # ⚠️ 2026-08-09 実測: 6種全部を有効にすると悪化した。
    #    ローカルPARC 95%→85%、RMSジャーク 15.88→20.17(+27%)、姿勢変化量 +26%。
    #    Colab の LeRobot 評価でも ベース90%→70%。独立した2測定が一致。
    #    ジャーク悪化は汎化では説明がつかない＝動きそのものが雑になった。
    #    犯人として疑わしいのは affine（画像と行動の対応をずらす）と
    #    hue（LIBERO は色で物体を識別する）。試すなら3種に絞ること:
    #      "--dataset.image_transforms.enable=true",
    #      "--dataset.image_transforms.tfs.hue.weight=0.0",
    #      "--dataset.image_transforms.tfs.affine.weight=0.0",
    #      "--dataset.image_transforms.tfs.saturation.weight=0.0",
    f"--dataset.image_transforms.enable={'true' if USE_AUGMENTATION else 'false'}",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_libero_plus_t1_lora_v6",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=2",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={SAVE_FREQ}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = str(LEROBOT_SRC) + os.pathsep + training_env.get("PYTHONPATH", "")
training_env["ACCELERATE_MIXED_PRECISION"] = MIXED_PRECISION
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print(f"学習開始: {STEPS:,} steps × batch {BATCH_SIZE} = {STEPS*BATCH_SIZE:,} サンプル")
print("（データの初回ダウンロードに時間がかかります）\n")

started = time.time()
process = subprocess.Popen(
    command, cwd=LEROBOT_DIR, env=training_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

recent_lines: deque[str] = deque(maxlen=80)
assert process.stdout is not None

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()
    if not line:
        continue
    recent_lines.append(line)
    if "step:" in line and "loss:" in line:
        step_match = re.search(r"step:\s*([0-9KM.]+)", line)
        loss_match = re.search(r"loss:([0-9.eE+-]+)", line)
        lr_match = re.search(r"lr:([0-9.eE+-]+)", line)
        elapsed = time.time() - started
        print(f"  step {step_match.group(1) if step_match else '?':>7}/{STEPS}  "
              f"loss={loss_match.group(1) if loss_match else 'n/a':>9}  "
              f"lr={lr_match.group(1) if lr_match else 'n/a':>9}  "
              f"経過 {elapsed/60:5.1f}分")

return_code = process.wait()
if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(f"Training failed: {return_code}")

print(f"\n学習完了。所要 {(time.time()-started)/60:.1f} 分")

## 9. LoRA をマージしてモデル全体を保存する

In [ ]:
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

checkpoint_dir = OUTPUT_DIR / "checkpoints" / f"{STEPS:06d}" / "pretrained_model"
if not (checkpoint_dir / "adapter_model.safetensors").is_file():
    found = sorted((OUTPUT_DIR / "checkpoints").glob("*/pretrained_model"))
    raise FileNotFoundError(f"アダプタが見つかりません。存在するのは: {found}")

gc.collect(); torch.cuda.empty_cache()

merge_config = PreTrainedConfig.from_pretrained(checkpoint_dir)
merge_config.device = "cpu"
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft = False

quiet = io.StringIO()
with contextlib.redirect_stdout(quiet), contextlib.redirect_stderr(quiet):
    base_policy = SmolVLAPolicy.from_pretrained(BASE_MODEL_LOCAL, config=merge_config, strict=False)
    peft_policy = PeftModel.from_pretrained(base_policy, checkpoint_dir,
                                            is_trainable=False, torch_device="cpu")
    merged_policy = peft_policy.merge_and_unload(safe_merge=True)

shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO
merged_policy.save_pretrained(MERGED_MODEL_DIR)

for pattern in ["policy_preprocessor.json", "policy_preprocessor*.safetensors",
                "policy_postprocessor.json", "policy_postprocessor*.safetensors"]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(source_path, MERGED_MODEL_DIR / source_path.name)

with safe_open(MERGED_MODEL_DIR / "model.safetensors", framework="pt", device="cpu") as w:
    keys = list(w.keys())
    if any("lora_" in k.lower() for k in keys):
        raise RuntimeError("マージ後も LoRA パラメータが残っています。")

# ★ PARC の2カメラ構成になっているかを確認する
merged_cfg = json.loads((MERGED_MODEL_DIR / "config.json").read_text(encoding="utf-8"))
img_feats = [k for k in merged_cfg["input_features"] if "images" in k]
print(f"重みキー数    : {len(keys)}  / lora_ を含む: 0  ✅")
print(f"input_features: {img_feats}")
print(f"empty_cameras : {merged_cfg.get('empty_cameras')}")
if len(img_feats) != 2 or merged_cfg.get("empty_cameras") not in (0, None):
    raise RuntimeError("カメラ構成が2台になっていません。PARC の観測と合いません。")
print("\n✅ 2カメラ構成 — PARC の観測と一致")

del peft_policy, base_policy, merged_policy
gc.collect(); torch.cuda.empty_cache()
print("Merged model ready.")

## 14. 保存してダウンロードする

`merged_model/` の中身を、そのまま提出物の `submission_template/model_weights/` に置き換えて使う。

**⚠️ 提出前に必ずローカル（Docker）で PARC の評価パイプラインを回すこと。**
このノートブックの評価は LeRobot の評価器であって、PARC の採点器ではない
（PARC は `成功 = 課題達成 かつ 対象外物体の変位 < 1mm`）。

In [ ]:
from zipfile import ZIP_STORED, ZipFile

if MERGED_ZIP_PATH.exists():
    MERGED_ZIP_PATH.unlink()

with ZipFile(MERGED_ZIP_PATH, mode="w", compression=ZIP_STORED, allowZip64=True) as archive:
    for file_path in sorted(MERGED_MODEL_DIR.rglob("*")):
        if file_path.is_file():
            archive.write(file_path, arcname=str(
                Path("model_weights") / file_path.relative_to(MERGED_MODEL_DIR)))

print(f"作成: {MERGED_ZIP_PATH}  ({MERGED_ZIP_PATH.stat().st_size/1e6:.1f} MB)")
print("展開すると model_weights/ になります。提出物の同名フォルダと差し替えてください。")

# Google ドライブにも保存する（セッションが切れても残る）
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/parc2026")
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MERGED_ZIP_PATH, drive_dir / MERGED_ZIP_PATH.name)
    shutil.copy2(COMPARISON_CSV_PATH, drive_dir / "comparison_t1.csv")
    # LoRAアダプタと学習状態も残す（再開できるように）
    if checkpoint_dir.exists():
        shutil.copytree(checkpoint_dir, drive_dir / "t1_lora_checkpoint", dirs_exist_ok=True)
    print(f"ドライブに保存: {drive_dir}")
except Exception as error:
    print(f"ドライブ保存はスキップ: {error}")

try:
    from google.colab import files
    files.download(str(MERGED_ZIP_PATH))
    files.download(str(COMPARISON_CSV_PATH))
except Exception as error:
    print(f"直接ダウンロードはスキップ: {error}")

---

## 次にやること（ローカル）

```bash
# 1. zip を展開して差し替え
cd C:\Users\odeod\dev\PARC2026_pre\submission_template
mv model_weights model_weights_prev
# ダウンロードした t1_lora_merged.zip を展開 → model_weights/ を置く

# 2. Docker で PARC の評価パイプラインを回す（本物の採点器）
docker run -it --rm -v "C:\Users\odeod\dev\PARC2026_pre:/host" parc2026:ready

#   コンテナ内
python /host/submission_template/policy_server.py --port 8000 > /tmp/server.log 2>&1 &
until curl -s http://localhost:8000/health > /dev/null 2>&1; do printf '.'; sleep 5; done

cd /workspace
python -m pipeline --server-url http://localhost:8000 --track track1 \
    --timeout 3600 --n-episodes 3 --max-steps 600 \
    --output-dir /host/results_new
```

**⚠️ 1エピソードで判断しないこと。** SmolVLA は flow matching で毎チャンク乱数を引くため、
同じモデル・同じ環境でも結果が変わる。最低3エピソード、できれば5エピソード。

## 覚えておくこと

| | |
|---|---|
| **スコア＝成功率そのもの** | ジャーク・SPARC・軌道長は総合スコアに入らない（`scorer.py:130`） |
| **成功 = 課題達成 かつ 衝突なし** | 対象外の物体が 1mm 動いたら無効 |
| **画像は180度回転** | 縦だけでは足りない。`arr[::-1, ::-1]` |
| **本番は1タスク20エピソード** | 達成率8%でも、1回以上達成する確率は81% |

詳細は `dev/PARC2026_pre/VLAとMyPolicyの仕組み.md` と `PARC2026_作業ログ.md`。